# 🔗 Module 2 — Neural Networks & Backpropagation
### Introduction to Deep Learning | AgenticLabs.ng

---

## 🎯 Learning Objectives
- Understand the architecture of a multi-layer neural network (MLP)
- Learn all key activation functions and when to use each
- Derive and implement **backpropagation** step by step
- Build and train an MLP in **PyTorch** on a real dataset (MNIST)
- Compare optimisers: SGD vs Adam vs RMSprop
- Read and interpret training/validation loss curves

---

## 2.1 — From Perceptron to Multi-Layer Network

A single perceptron cannot solve non-linear problems (e.g. XOR). The solution is to stack multiple layers:

```
Input → [Hidden Layer 1] → [Hidden Layer 2] → ... → Output
         (linear + activation)  (linear + activation)
```

Each hidden layer learns increasingly abstract representations. This is the key idea of **deep** learning.


## 2.2 — Activation Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ── All major activation functions ────────────────────────
x = np.linspace(-4, 4, 300)

activations = {
    "Sigmoid": (1/(1+np.exp(-x)),     "Saturates for |x|>2. Good for output (binary). Not for hidden layers."),
    "Tanh":    (np.tanh(x),           "Zero-centred sigmoid. Better than sigmoid for hidden layers."),
    "ReLU":    (np.maximum(0, x),     "Most popular. Fast. Can suffer 'dying ReLU' problem."),
    "Leaky ReLU": (np.where(x>=0, x, 0.1*x), "Fixes dying ReLU by allowing small negative slope."),
    "ELU":     (np.where(x>=0, x, 0.5*(np.exp(x)-1)), "Smooth, negative saturation. Often beats ReLU."),
    "GELU":    (x * (1/(1+np.exp(-1.702*x))),          "Used in BERT, GPT. Smooth probabilistic gating."),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()
colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860']

for i, (name, (vals, desc)) in enumerate(activations.items()):
    axes[i].plot(x, vals, color=colors[i], linewidth=2.5)
    axes[i].axhline(0, color='gray', linewidth=0.7, linestyle='--')
    axes[i].axvline(0, color='gray', linewidth=0.7, linestyle='--')
    axes[i].set_title(name, fontsize=13, fontweight='bold')
    axes[i].set_xlabel(desc, fontsize=8.5, color='#555')
    axes[i].grid(alpha=0.3)
    axes[i].set_ylim(-1.2, 2)

plt.suptitle("Activation Functions Compared", fontsize=14)
plt.tight_layout(); plt.show()


## 2.3 — Backpropagation: The Chain Rule

Backpropagation is just the **chain rule of calculus** applied to a computation graph. It computes the gradient of the loss with respect to every parameter.

**Forward pass**: compute predictions  
**Backward pass**: compute gradients layer by layer, from output → input

```
∂L/∂w1 = (∂L/∂y_hat) × (∂y_hat/∂h) × (∂h/∂w1)
```

Let's implement a 2-layer MLP from scratch to see this clearly.


In [ ]:
# ── 2-Layer MLP from scratch (NumPy) ─────────────────────
class MLP_Scratch:
    """2-layer MLP: input → hidden (ReLU) → output (sigmoid)"""
    
    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.lr = lr
        # He initialisation (good for ReLU)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2/input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2/hidden_size)
        self.b2 = np.zeros((1, output_size))
    
    def relu(self, z):        return np.maximum(0, z)
    def sigmoid(self, z):     return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    def relu_grad(self, z):   return (z > 0).astype(float)
    
    def forward(self, X):
        self.X  = X
        self.z1 = X @ self.W1 + self.b1
        self.h1 = self.relu(self.z1)
        self.z2 = self.h1 @ self.W2 + self.b2
        self.y_hat = self.sigmoid(self.z2)
        return self.y_hat
    
    def loss(self, y_hat, y):
        eps = 1e-8
        return -np.mean(y*np.log(y_hat+eps) + (1-y)*np.log(1-y_hat+eps))
    
    def backward(self, y):
        n = len(y)
        # Output layer gradients
        dL_dyhat = (self.y_hat - y) / n
        dL_dz2   = dL_dyhat * self.y_hat * (1 - self.y_hat)
        dL_dW2   = self.h1.T @ dL_dz2
        dL_db2   = np.sum(dL_dz2, axis=0, keepdims=True)
        # Hidden layer gradients (chain rule)
        dL_dh1   = dL_dz2 @ self.W2.T
        dL_dz1   = dL_dh1 * self.relu_grad(self.z1)
        dL_dW1   = self.X.T @ dL_dz1
        dL_db1   = np.sum(dL_dz1, axis=0, keepdims=True)
        # Update parameters
        self.W2 -= self.lr * dL_dW2
        self.b2 -= self.lr * dL_db2
        self.W1 -= self.lr * dL_dW1
        self.b1 -= self.lr * dL_db1

# ── XOR problem (impossible for single perceptron!) ────────
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([[0],[1],[1],[0]], dtype=float)

mlp = MLP_Scratch(input_size=2, hidden_size=8, output_size=1, lr=0.5)
losses = []
for epoch in range(5000):
    y_hat = mlp.forward(X_xor)
    loss  = mlp.loss(y_hat, y_xor)
    mlp.backward(y_xor)
    losses.append(loss)

preds = (mlp.forward(X_xor) > 0.5).astype(int)
print("XOR problem — Predictions vs Truth:")
for xi, yi, pi in zip(X_xor, y_xor, preds):
    print(f"  Input: {xi.astype(int)} | Expected: {int(yi[0])} | Got: {pi[0]}")

plt.figure(figsize=(6, 3))
plt.plot(losses, color='#4C72B0', linewidth=1.5)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("XOR — Training Loss")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 2.4 — Building an MLP in PyTorch

Now we use PyTorch to build the same network more cleanly, and apply it to the real MNIST handwritten digit dataset.

In [ ]:
# ── Load MNIST ──────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))   # MNIST mean & std
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Training samples : {len(train_dataset):,}")
print(f"Test samples     : {len(test_dataset):,}")
print(f"Image shape      : {train_dataset[0][0].shape}")
print(f"Number of classes: 10  (digits 0-9)")

# Show sample images
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.ravel()):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')
plt.suptitle("MNIST Sample Images (28×28 pixels)", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── Define the MLP in PyTorch ─────────────────────────────
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),           # 28×28 → 784
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)       # 10 classes
        )
    
    def forward(self, x):
        return self.network(x)

model = MLP().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ── Training loop: compare SGD vs Adam ────────────────────
def train_model(model, optimiser, epochs=5):
    criterion = nn.CrossEntropyLoss()
    train_losses, val_accuracies = [], []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimiser.zero_grad()
            out  = model(X_batch)
            loss = criterion(out, y_batch)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()
        
        # Evaluate on test set
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for X_b, y_b in test_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                preds = model(X_b).argmax(dim=1)
                correct += (preds == y_b).sum().item()
                total   += y_b.size(0)
        
        avg_loss = epoch_loss / len(train_loader)
        acc      = correct / total * 100
        train_losses.append(avg_loss)
        val_accuracies.append(acc)
        print(f"  Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")
    
    return train_losses, val_accuracies

# Train with SGD
print("\n── SGD Optimiser ─────────────────────────────────")
model_sgd  = MLP().to(device)
opt_sgd    = optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)
sgd_losses, sgd_accs = train_model(model_sgd, opt_sgd, epochs=5)

# Train with Adam
print("\n── Adam Optimiser ────────────────────────────────")
model_adam = MLP().to(device)
opt_adam   = optim.Adam(model_adam.parameters(), lr=0.001)
adam_losses, adam_accs = train_model(model_adam, opt_adam, epochs=5)


In [ ]:
# ── Plot training comparison ──────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(sgd_losses,  'b-o', label='SGD',  markersize=6)
ax1.plot(adam_losses, 'r-o', label='Adam', markersize=6)
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training Loss: SGD vs Adam"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(sgd_accs,  'b-o', label='SGD',  markersize=6)
ax2.plot(adam_accs, 'r-o', label='Adam', markersize=6)
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Test Accuracy (%)")
ax2.set_title("Test Accuracy: SGD vs Adam"); ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle("MNIST MLP — SGD vs Adam", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ── Confusion matrix ──────────────────────────────────────
from sklearn.metrics import confusion_matrix
import seaborn as sns

model_adam.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        preds = model_adam(X_b).argmax(dim=1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(y_b.numpy())

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix — MNIST MLP (Adam)"); plt.show()
print(f"\nOverall accuracy: {np.diag(cm).sum()/cm.sum()*100:.2f}%")


## 📝 Exercises

1. **Add more layers**: Add two more hidden layers to `MLP`. Does accuracy improve?
2. **Activation comparison**: Replace `ReLU` with `nn.Tanh()` in all hidden layers. Which performs better after 5 epochs?
3. **Learning rate sensitivity**: Train with Adam at `lr=0.1` and `lr=0.00001`. What do the loss curves look like?
4. **Worst predictions**: Find the 9 images that the model misclassified with highest confidence. What do they look like?

---

## ✅ Module 2 Summary

| Concept | Key takeaway |
|---|---|
| Multi-layer network | Stacking layers allows learning non-linear features |
| ReLU | The most common activation — fast and effective |
| Backpropagation | Chain rule applied backward through the graph |
| CrossEntropyLoss | Standard loss for multi-class classification |
| Adam | Adaptive optimiser — usually the safe default choice |

**Next up → Module 3: Convolutional Neural Networks**
